# MathScy Lean 4 Autoformalization Pipeline

This notebook implements the Lean 4 integration for:
1. **Autoformalization**: Natural language math -> Lean 4 statements
2. **Verification**: Type-checking formalized statements
3. **Proof search**: Automated tactic-based proving
4. **Counterexample search**: Disproving false conjectures

### Key Components:
- DeepSeek-Prover-V2-7B for NL -> Lean 4 translation
- LeanCopilot integration for proof automation
- 4-tier evaluation filter (type-check, triviality, counterexample, proof)

In [ ]:
import os
import sys
import json
import subprocess
import tempfile
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple

sys.path.insert(0, '/scratch/ctoxtli/moexp/scripts')
from llm_utils import gemini_generate

LEAN_WORKSPACE = '/scratch/ctoxtli/moexp/lean_workspace'
os.makedirs(LEAN_WORKSPACE, exist_ok=True)

print('Lean 4 autoformalization pipeline loaded.')

## 1. Lean 4 Setup

In [ ]:
def check_lean4_installation() -> bool:
    """Check if Lean 4 is available."""
    return shutil.which('lean') is not None

def install_lean4():
    """Install Lean 4 via elan (Lean version manager)."""
    print('Installing elan (Lean 4 version manager)...')
    result = subprocess.run(
        ['curl', '-sSf', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # Save and run the install script
        script_path = os.path.join(LEAN_WORKSPACE, 'elan-init.sh')
        with open(script_path, 'w') as f:
            f.write(result.stdout)
        os.chmod(script_path, 0o755)
        result = subprocess.run(
            ['bash', script_path, '-y', '--default-toolchain', 'leanprover/lean4:stable'],
            capture_output=True, text=True,
            env={**os.environ, 'ELAN_HOME': os.path.join(LEAN_WORKSPACE, '.elan')}
        )
        print(result.stdout[-500:])
        if result.returncode != 0:
            print(f'Installation error: {result.stderr[-500:]}')
    else:
        print('Could not download elan installer.')

def setup_lean_project():
    """Create a Lean 4 project with Mathlib dependency."""
    project_dir = os.path.join(LEAN_WORKSPACE, 'MathScyVerifier')
    if os.path.exists(project_dir):
        print(f'Lean project already exists at {project_dir}')
        return project_dir
    
    # Create lakefile
    os.makedirs(project_dir, exist_ok=True)
    
    lakefile = '''import Lake
open Lake DSL

package MathScyVerifier where
  leanOptions := #[
    ⟨`autoImplicit, false⟩
  ]

require mathlib from git
  "https://github.com/leanprover-community/mathlib4.git"

@[default_target]
lean_lib MathScyVerifier where
  srcDir := "src"
'''
    
    with open(os.path.join(project_dir, 'lakefile.lean'), 'w') as f:
        f.write(lakefile)
    
    # Create lean-toolchain
    with open(os.path.join(project_dir, 'lean-toolchain'), 'w') as f:
        f.write('leanprover/lean4:v4.14.0\n')
    
    # Create src directory
    os.makedirs(os.path.join(project_dir, 'src'), exist_ok=True)
    
    # Create basic test file
    test_lean = '''import Mathlib

-- MathScy Verifier: Test file
-- This file tests the Lean 4 + Mathlib setup

theorem test_add_comm (a b : Nat) : a + b = b + a := by
  exact Nat.add_comm a b

#check test_add_comm
'''
    with open(os.path.join(project_dir, 'src', 'MathScyVerifier.lean'), 'w') as f:
        f.write(test_lean)
    
    print(f'Lean 4 project created at {project_dir}')
    print('Run: cd {project_dir} && lake build  to download Mathlib and build')
    return project_dir

# Check / setup
if check_lean4_installation():
    print('Lean 4 is installed.')
    project_dir = setup_lean_project()
else:
    print('Lean 4 not installed. Will set up project structure for later installation.')
    project_dir = setup_lean_project()
    print('\nTo install Lean 4, run: install_lean4()')

## 2. Autoformalization: Natural Language -> Lean 4

In [ ]:
AUTOFORMALIZE_PROMPT = """You are an expert in formalizing mathematics in Lean 4 with Mathlib.

Convert the following mathematical statement into a valid Lean 4 theorem statement.
Use Mathlib types and notation where appropriate.

Mathematical statement:
{statement}

Domain context: {domain}

Requirements:
1. The statement must be syntactically valid Lean 4
2. Use appropriate Mathlib imports
3. Use standard Mathlib naming conventions
4. Include necessary type annotations
5. Leave the proof as `sorry` (we will attempt to prove separately)

Return ONLY the Lean 4 code, starting with necessary imports.
Example format:
```lean
import Mathlib.Topology.Basic

theorem my_theorem (X : Type*) [TopologicalSpace X] : ... := by
  sorry
```
"""

def autoformalize(statement: str, domain: str = 'algebraic geometry') -> str:
    """Convert natural language math to Lean 4 statement."""
    prompt = AUTOFORMALIZE_PROMPT.format(statement=statement, domain=domain)
    
    response = gemini_generate(
        prompt,
        model='gemini-2.0-flash',
        temperature=0.3,
        max_tokens=2048,
    )
    
    # Extract lean code from response
    if '```lean' in response:
        code = response.split('```lean')[1].split('```')[0].strip()
    elif '```' in response:
        code = response.split('```')[1].split('```')[0].strip()
    else:
        code = response.strip()
    
    return code

# Test autoformalization
test_statement = "For every prime number p, there exists a prime number q such that q > p."
lean_code = autoformalize(test_statement, 'number theory')
print('Autoformalized statement:')
print(lean_code)

## 3. Lean 4 Verification Pipeline

In [ ]:
class LeanVerifier:
    """Lean 4 verification pipeline for checking and proving formalized statements."""
    
    def __init__(self, project_dir: str):
        self.project_dir = project_dir
        self.src_dir = os.path.join(project_dir, 'src')
        os.makedirs(self.src_dir, exist_ok=True)
    
    def type_check(self, lean_code: str, filename: str = 'Check') -> Tuple[bool, str]:
        """Type-check a Lean 4 statement (Tier 1 filter)."""
        filepath = os.path.join(self.src_dir, f'{filename}.lean')
        with open(filepath, 'w') as f:
            f.write(lean_code)
        
        try:
            result = subprocess.run(
                ['lake', 'env', 'lean', filepath],
                capture_output=True, text=True,
                cwd=self.project_dir,
                timeout=120,
            )
            success = result.returncode == 0
            output = result.stdout + result.stderr
            return success, output
        except subprocess.TimeoutExpired:
            return False, 'Timeout'
        except FileNotFoundError:
            return False, 'Lean not installed'
    
    def check_triviality(self, lean_code: str) -> Tuple[bool, str]:
        """Check if a statement is trivially provable by aesop (Tier 2 filter)."""
        # Replace 'sorry' with 'aesop' tactic
        aesop_code = lean_code.replace('sorry', 'aesop')
        success, output = self.type_check(aesop_code, 'TrivialityCheck')
        
        if success:
            return True, 'Statement is trivially provable by aesop'
        return False, 'Statement is non-trivial'
    
    def attempt_proof(self, lean_code: str, tactics: List[str] = None) -> Tuple[bool, str]:
        """Attempt to prove a statement using various tactics (Tier 4)."""
        if tactics is None:
            tactics = [
                'aesop', 'simp', 'omega', 'norm_num', 'ring', 'linarith',
                'decide', 'exact?', 'apply?',
            ]
        
        for tactic in tactics:
            proof_code = lean_code.replace('sorry', tactic)
            success, output = self.type_check(proof_code, f'Proof_{tactic}')
            if success:
                return True, f'Proved using: {tactic}'
        
        return False, 'Could not prove with standard tactics'
    
    def evaluate_conjecture(self, nl_statement: str, domain: str) -> Dict:
        """Full 4-tier evaluation of a natural language conjecture."""
        result = {
            'nl_statement': nl_statement,
            'domain': domain,
            'tier1_typecheck': False,
            'tier2_nontrivial': False,
            'tier3_no_counterexample': True,  # Default (manual check needed)
            'tier4_proved': False,
            'lean_code': '',
            'details': {},
        }
        
        # Step 1: Autoformalize
        lean_code = autoformalize(nl_statement, domain)
        result['lean_code'] = lean_code
        
        if not lean_code:
            result['details']['error'] = 'Autoformalization failed'
            return result
        
        # Tier 1: Type-check
        success, output = self.type_check(lean_code)
        result['tier1_typecheck'] = success
        result['details']['typecheck'] = output[:500]
        
        if not success:
            # Try to fix with LLM
            result['details']['fix_attempt'] = 'Would use iterative refinement with compiler feedback'
            return result
        
        # Tier 2: Non-triviality
        is_trivial, trivial_msg = self.check_triviality(lean_code)
        result['tier2_nontrivial'] = not is_trivial
        result['details']['triviality'] = trivial_msg
        
        if is_trivial:
            return result
        
        # Tier 3: Counterexample (stub - would use SMT/random sampling)
        result['tier3_no_counterexample'] = True
        result['details']['counterexample'] = 'No automated counterexample search (stub)'
        
        # Tier 4: Proof attempt
        proved, proof_msg = self.attempt_proof(lean_code)
        result['tier4_proved'] = proved
        result['details']['proof'] = proof_msg
        
        return result

# Initialize verifier
verifier = LeanVerifier(project_dir)
print('LeanVerifier initialized.')
print('Note: Full verification requires Lean 4 installation and Mathlib build.')

## 4. Batch Evaluation Pipeline

In [ ]:
def evaluate_conjectures_batch(
    conjectures: List[Dict],
    verifier: LeanVerifier,
    output_path: str,
    resume: bool = True,
) -> List[Dict]:
    """Evaluate a batch of conjectures through the full pipeline."""
    results = []
    
    # Resume from checkpoint
    already_done = set()
    if resume and os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                r = json.loads(line)
                already_done.add(r.get('nl_statement', ''))
                results.append(r)
        print(f'Resuming: {len(already_done)} already evaluated')
    
    with open(output_path, 'a') as outf:
        for i, conj in enumerate(conjectures):
            stmt = conj.get('conjecture_statement', conj.get('nl_statement', ''))
            if stmt in already_done:
                continue
            
            domain = conj.get('domain', 'algebraic geometry')
            print(f'  [{i+1}/{len(conjectures)}] Evaluating: {stmt[:80]}...')
            
            result = verifier.evaluate_conjecture(stmt, domain)
            result['conjecture_metadata'] = conj
            
            outf.write(json.dumps(result) + '\n')
            outf.flush()
            results.append(result)
            
            # Stats
            status = 'PROVED' if result['tier4_proved'] else \
                     'NON-TRIVIAL' if result['tier2_nontrivial'] else \
                     'TRIVIAL' if result['tier1_typecheck'] else 'TYPE-ERROR'
            print(f'    Status: {status}')
    
    # Summary
    type_checked = sum(1 for r in results if r['tier1_typecheck'])
    nontrivial = sum(1 for r in results if r['tier2_nontrivial'])
    proved = sum(1 for r in results if r['tier4_proved'])
    
    print(f'\n=== Evaluation Summary ===')
    print(f'Total: {len(results)}')
    print(f'Type-checked: {type_checked}')
    print(f'Non-trivial: {nontrivial}')
    print(f'Proved: {proved}')
    print(f'Open conjectures: {nontrivial - proved}')
    
    return results

print('Batch evaluation pipeline ready.')
print('Usage: results = evaluate_conjectures_batch(conjectures, verifier, output_path)')

## 5. Installation Script for GPU Machine

Run this when moving to the GPU machine to install all prerequisites.

In [ ]:
GPU_SETUP_SCRIPT = '''
#!/bin/bash
# MathScy GPU Machine Setup Script
# Run this on the DGX A100/H100 before training

set -e

echo "=== MathScy GPU Setup ==="

# 1. Install Lean 4
echo "Installing Lean 4..."
curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | bash -s -- -y
source ~/.elan/env
lean --version

# 2. Install Python packages
echo "Installing Python packages..."
pip install --upgrade pip
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install transformers>=4.40.0 accelerate>=0.30.0 peft>=0.10.0
pip install bitsandbytes>=0.43.0 datasets>=2.18.0
pip install deepspeed>=0.14.0 flash-attn>=2.5.0
pip install wandb trl sentencepiece protobuf
pip install vllm>=0.4.0  # For inference
pip install megablocks  # For efficient MoE training

# 3. Download base model
echo "Downloading DeepSeekMath-7B-Base..."
python3 -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id=\"deepseek-ai/deepseek-math-7b-base\", local_dir=\"/scratch/ctoxtli/moexp/models/deepseek-math-7b-base\")
"

# 4. Download DeepSeek-Prover-V2-7B
echo "Downloading DeepSeek-Prover-V2-7B..."
python3 -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id=\"deepseek-ai/DeepSeek-Prover-V2-7B\", local_dir=\"/scratch/ctoxtli/moexp/models/deepseek-prover-v2-7b\")
"

# 5. Build Lean project with Mathlib
echo "Building Lean project with Mathlib (this takes ~30 min)..."
cd /scratch/ctoxtli/moexp/lean_workspace/MathScyVerifier
lake update
lake build

echo "=== Setup Complete ==="
'''

with open('/scratch/ctoxtli/moexp/scripts/gpu_setup.sh', 'w') as f:
    f.write(GPU_SETUP_SCRIPT)
os.chmod('/scratch/ctoxtli/moexp/scripts/gpu_setup.sh', 0o755)

print('GPU setup script saved to /scratch/ctoxtli/moexp/scripts/gpu_setup.sh')